# Tübingen Bicycle Counter Analysis

This notebook demonstrates the new modular tuecycle package for analyzing bike counter data with weather correlations.

## Quick Start

```python
from tuecycle import DataManager
from tuecycle.plots import get_plot, list_plots

dm = DataManager()
df = dm.get("tuebingen_tunnel")
fig = get_plot("hourly_pattern")(df, "Tübingen Radtunnel")
fig.show()
```

## 1. Setup and Imports

First, let's import the tuecycle package and see what's available.

In [ ]:
# Import the tuecycle package
from tuecycle import DataManager, get_station
from tuecycle.config import list_stations
from tuecycle.plots import get_plot, list_plots

# Show available stations
print("Available stations:")
for alias in list_stations():
    station = get_station(alias)
    print(f"  • {alias}: {station.display_name} ({station.city})")

ImportError: cannot import name 'get_plot' from 'tuecycle.config.stations' (/Users/laurin/Desktop/tuecycle/tuecycle/config/stations.py)

In [2]:
# Show available plots
print("\nAvailable plots:")
for name in list_plots():
    print(f"  • {name}")


Available plots:
  • {'name': 'bike_vs_temp_heatmap', 'description': '2D density heatmap of bike counts vs temperature'}
  • {'name': 'city_comparison_hourly', 'description': 'Compare hourly patterns across stations'}
  • {'name': 'city_comparison_monthly', 'description': 'Compare monthly patterns across stations (normalized)'}
  • {'name': 'fourier_transform', 'description': 'Fourier transform showing periodic patterns'}
  • {'name': 'hour_weekday_heatmap', 'description': 'Heatmap of bike counts by hour and day of week'}
  • {'name': 'hourly_pattern', 'description': 'Average bike count by hour of day with std band'}
  • {'name': 'monthly_average', 'description': 'Bar chart of average bike counts per month'}
  • {'name': 'rush_hour_analysis', 'description': 'Bar chart of bike counts by time category'}
  • {'name': 'seasonal_comparison', 'description': 'Hourly pattern by season (Winter/Transition/Summer)'}
  • {'name': 'temp_deviation_heatmap', 'description': 'Temperature deviation vs 

## 2. Load Data with Parquet Caching

The `DataManager` handles loading bike counter data merged with weather data. On first run, it loads from CSVs and caches to Parquet. Subsequent loads are **10-50x faster**.

In [3]:
import time

# Initialize the DataManager
dm = DataManager()

# Load data for Tübingen (first run may take a few seconds, subsequent loads are instant)
start = time.time()
tue_df = dm.get("tuebingen_tunnel")
print(f"Loaded Tübingen data in {time.time() - start:.2f}s")
print(f"Shape: {tue_df.shape}")
print(f"Columns: {list(tue_df.columns)}")
tue_df.head()

Loaded Tübingen data in 0.12s
Shape: (8759, 4)
Columns: ['datetime', 'bike', 'rain', 'temp']


,datetime,bike,rain,temp
0,2024-11-01 00:00:00,35.0,0.0,5.2
1,2024-11-01 01:00:00,45.0,0.0,5.4
2,2024-11-01 02:00:00,37.0,0.0,5.7
3,2024-11-01 03:00:00,19.0,0.0,5.7
4,2024-11-01 04:00:00,9.0,0.0,5.8


In [4]:
# Load multiple stations at once
start = time.time()
dfs = dm.get_multiple(["tuebingen_tunnel", "heidelberg_mannheimer", "mannheim_fernmeldeturm"])
print(f"Loaded 3 stations in {time.time() - start:.2f}s")

# Unpack for convenience
hd_df = dfs["heidelberg_mannheimer"]
ma_df = dfs["mannheim_fernmeldeturm"]

print(f"\nHeidelberg: {len(hd_df)} rows")
print(f"Mannheim: {len(ma_df)} rows")

Loaded 3 stations in 0.01s

Heidelberg: 8759 rows
Mannheim: 8759 rows


## 3. Single-Station Plots

### Time Series Overview
Interactive plot showing bike counts, temperature, and rainfall with range slider.

In [5]:
# Time series plot
fig = get_plot("time_series")(tue_df, title="Tübingen Radtunnel")
fig.show()

### Hourly Pattern
Average bike count for each hour of the day with ±1 standard deviation band.

In [6]:
fig = get_plot("hourly_pattern")(tue_df, title="Tübingen Radtunnel")
fig.show()

### Seasonal Comparison
Compare hourly patterns across:
- **Winter:** Nov, Dec, Jan, Feb
- **Transition:** Mar, Apr, Sep, Oct  
- **Summer:** May, Jun, Jul, Aug

In [7]:
fig = get_plot("seasonal_comparison")(tue_df, title="Tübingen Radtunnel")
fig.show()

### Weekday vs Weekend

In [8]:
fig = get_plot("weekday_vs_weekend")(tue_df, title="Tübingen Radtunnel")
fig.show()

### Hour × Weekday Heatmap

In [9]:
fig = get_plot("hour_weekday_heatmap")(tue_df, title="Tübingen Radtunnel")
fig.show()

### Monthly Average

In [10]:
fig = get_plot("monthly_average")(tue_df, title="Tübingen Radtunnel")
fig.show()

### Bike vs Temperature Heatmap
2D density showing relationship between temperature and bike counts (daytime hours only).

In [11]:
fig = get_plot("bike_vs_temp_heatmap")(tue_df, title="Tübingen Radtunnel")
fig.show()

### Temperature Deviation Heatmap
Removes seasonal bias by showing how temperature deviation from monthly average affects bike counts.
- **Slope** quantifies sensitivity: % change in bike count per °C deviation

In [12]:
fig = get_plot("temp_deviation_heatmap")(tue_df, title="Tübingen Radtunnel")
fig.show()

### Rush Hour Analysis
Compare average bike counts across time categories:
- Morning Rush (7-9 AM weekdays)
- Evening Rush (5-7 PM weekdays)
- Weekday Non-Rush
- Weekend

In [13]:
fig = get_plot("rush_hour_analysis")(tue_df, title="Tübingen Radtunnel")
fig.show()

### Fourier Transform Analysis
Frequency spectrum of bike count signal reveals periodic patterns:
- **Daily cycle** (1/24 Hz): Main commute pattern
- **12h cycle**: Morning + evening rush peaks
- **Weekly cycle** (1/168 Hz): Weekday vs weekend differences

In [14]:
fig = get_plot("fourier_transform")(tue_df, title="Tübingen Radtunnel - Bike Counts")
fig.show()

## 4. Multi-Station Comparison Plots

These plots compare patterns across multiple cities.

In [17]:
# City comparison requires a dict of {station_alias: dataframe}
city_data = {
    "tuebingen_tunnel": tue_df,
    "heidelberg_mannheimer": hd_df,
    "mannheim_fernmeldeturm": ma_df
}

### Hourly Pattern Comparison

In [18]:
fig = get_plot("city_comparison_hourly")(city_data)
fig.show()

### Monthly Pattern Comparison (Normalized)

In [19]:
fig = get_plot("city_comparison_monthly")(city_data)
fig.show()

### Winter to Summer Ratio
Shows seasonal swing by hour:
- **100%** = Winter equals Summer
- **50%** = Summer has 2x bikes of Winter

In [20]:
fig = get_plot("winter_summer_ratio")(city_data)
fig.show()

## 5. Interactive Dashboard

All plots accessible in one unified interface with dropdown menus.

In [21]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Available plot types
single_station_plots = [
    "time_series",
    "hourly_pattern", 
    "seasonal_comparison",
    "weekday_vs_weekend",
    "hour_weekday_heatmap",
    "monthly_average",
    "bike_vs_temp_heatmap",
    "temp_deviation_heatmap",
    "rush_hour_analysis",
    "fourier_transform"
]

multi_station_plots = [
    "city_comparison_hourly",
    "city_comparison_monthly", 
    "winter_summer_ratio"
]

# Create widgets
station_dropdown = widgets.Dropdown(
    options=list_stations(),
    value="tuebingen_tunnel",
    description="Station:",
    style={"description_width": "60px"}
)

plot_dropdown = widgets.Dropdown(
    options=single_station_plots + ["---Multi-Station---"] + multi_station_plots,
    value="hourly_pattern",
    description="Plot:",
    style={"description_width": "60px"}
)

output = widgets.Output()

def update_plot(change=None):
    with output:
        clear_output(wait=True)
        plot_name = plot_dropdown.value
        
        if plot_name.startswith("---"):
            print("Select a plot type")
            return
            
        plot_fn = get_plot(plot_name)
        
        if plot_name in multi_station_plots:
            # Multi-station plot
            fig = plot_fn(city_data)
        else:
            # Single-station plot
            station = get_station(station_dropdown.value)
            df = dm.get(station_dropdown.value)
            fig = plot_fn(df, title=station.display_name)
        
        fig.show()

# Connect callbacks
station_dropdown.observe(update_plot, names="value")
plot_dropdown.observe(update_plot, names="value")

# Display
display(widgets.HBox([station_dropdown, plot_dropdown]))
display(output)

# Initial plot
update_plot()

Output()

## 6. Adding New Stations

To add a new station, edit `tuecycle/config/stations.py`:

```python
STATIONS["my_new_station"] = Station(
    alias="my_new_station",
    city="CityName",  # Must match weather file name
    counter_name="Full Counter Name from CSV",
    display_name="Friendly Display Name",
    color="#HexColor"
)
```

Then load data as usual:
```python
df = dm.get("my_new_station")
```

## 7. Cache Management

The cache is stored in the `cache/` directory as Parquet files.

In [22]:
# List cached files
import os
cache_dir = "cache"
if os.path.exists(cache_dir):
    cached_files = os.listdir(cache_dir)
    print(f"Cached files ({len(cached_files)}):")
    for f in sorted(cached_files):
        size_mb = os.path.getsize(os.path.join(cache_dir, f)) / 1024 / 1024
        print(f"  • {f} ({size_mb:.2f} MB)")
else:
    print("No cache directory yet")

Cached files (3):
  • heidelberg_mannheimer_2024-11-01_2025-10-31.parquet (0.10 MB)
  • mannheim_fernmeldeturm_2024-11-01_2025-10-31.parquet (0.10 MB)
  • tuebingen_tunnel_2024-11-01_2025-10-31.parquet (0.10 MB)


In [ ]:
# To clear the cache (forces reload from CSVs):
# dm.clear_cache()

# To preload all stations:
# dm.preload_all()

# The dm object defines the date range of the loaded data!